# Chapter 13: Transformers


<!-- Macro definitions for MathJax, mirroring book.tex -->
$$
\newcommand{\bm}[1]{\boldsymbol{#1}}
\newcommand{\Det}[1]{|\boldsymbol{#1}|}
\newcommand{\bigO}{\mathcal{O}}
\newcommand{\var}{\mathrm{Var}}
\newcommand{\cov}{\mathrm{Cov}}
\newcommand{\Prob}{\mathrm{Prob}}
\newcommand{\mean}[1]{\langle #1 \rangle}
$$

Each architecture in this book has been a restriction of the dense layer,
justified by a property of the data.  Chapter 10 imposed locality
and translation equivariance and obtained convolution; Chapter 11
imposed sequential processing with a carried state and obtained recurrence.  In
each case a *fixed* interaction pattern was written into the architecture
before any data were seen: a convolutional layer couples a pixel to its
neighbours whatever the image contains, and a recurrent layer couples each step
to the previous one whatever the sequence says.

The transformer takes the opposite decision.  It couples every element to every
other element, and it lets the *strength* of each coupling be computed from
the data at run time.  The interaction pattern is not designed; it is inferred,
afresh for every input.  That single change of principle -- from fixed couplings
to *adaptive* couplings -- is what this chapter is about, and it turns out
to have consequences that are mathematically sharp, physically suggestive, and
occasionally limiting in ways worth knowing.

This chapter follows the transformer lecture notes for the course.  We derive
attention, prove the properties that characterise it, implement it from scratch
with verified gradients, and measure two things that matter: why the
$1/\sqrt{d_k}$ in Eq. (13.5) is not cosmetic, and what a single
attention layer provably cannot compute.


## Tokens

A transformer consumes a sequence of vectors,

$$
\bm{x}_1,\bm{x}_2,\dots,\bm{x}_n,
  \qquad \bm{x}_i\in\mathbb{R}^{d},\tag{13.1}
$$

called *tokens*, and returns another sequence of the same length,
$(\bm{y}_1,\dots,\bm{y}_n)$.  What a token *is* depends on the problem:
a word in a sentence, a patch of an image, a node of a discretised field, or a
sample of a solution to a partial differential equation.  Collect them as the
rows of $\bm{X}\in\mathbb{R}^{n\times d}$.

The generality is the point.  Anything that can be cut into pieces and given
a vector description can be handed to a transformer, and the architecture makes
no assumption about how the pieces are arranged.  As Theorem thm:13-perm will
prove, it makes *no* assumption about their order either, which is a
liberation and a problem in equal measure.


## Scaled dot-product attention

For each position $i$ we want an output that is a weighted combination of
information drawn from all positions,

$$
\bm{y}_i = \sum_{j=1}^{n} A_{ij}\,\bm{v}_j,\tag{13.2}
$$

with weights $A_{ij}$ that depend on the data.  Three learned linear maps
supply the ingredients.  From each token we form a *query*, a *key*
and a *value*,

$$
\bm{q}_i = \bm{W}_Q^{\mathsf{T}}\bm{x}_i,
  \qquad
  \bm{k}_j = \bm{W}_K^{\mathsf{T}}\bm{x}_j,
  \qquad
  \bm{v}_j = \bm{W}_V^{\mathsf{T}}\bm{x}_j,\tag{13.3}
$$

with $\bm{W}_Q,\bm{W}_K\in\mathbb{R}^{d\times d_k}$ and
$\bm{W}_V\in\mathbb{R}^{d\times d_v}$.  The reading is worth memorising: the
query is what position $i$ is looking for, the key is what position $j$
advertises, and the value is what position $j$ contributes if selected.

The weight is the softmax of the query-key inner product,

$$
A_{ij} = \frac{\exp(s_{ij})}{\sum_{\ell}\exp(s_{i\ell})},
  \qquad
  s_{ij} = \frac{\bm{q}_i\cdot\bm{k}_j}{\sqrt{d_k}} .\tag{13.4}
$$

In matrix form, with $\bm{Q}=\bm{X}\bm{W}_Q$, $\bm{K}=\bm{X}\bm{W}_K$ and
$\bm{V}=\bm{X}\bm{W}_V$,

$$
\boxed{\;
  \mathrm{Attention}(\bm{Q},\bm{K},\bm{V})
  = \mathrm{softmax}\!\left(\frac{\bm{Q}\bm{K}^{\mathsf{T}}}{\sqrt{d_k}}\right)
    \bm{V}. \;}\tag{13.5}
$$

The $n\times n$ matrix $\bm{Q}\bm{K}^{\mathsf{T}}$ holds the interaction between
every pair of positions; the softmax normalises each row; multiplication by
$\bm{V}$ aggregates.  When $\bm{Q}$, $\bm{K}$ and $\bm{V}$ all come from the
same $\bm{X}$, as here, this is *self*-attention.

### What attention is

Three statements characterise Eq. (13.5), and each has a
consequence that is easy to state and easy to forget.

```{admonition} Proposition (Attention averages)
:class: important
The matrix $\bm{A}$ of Eq. (13.4) is *row-stochastic*:
$A_{ij}>0$ and $\sum_j A_{ij}=1$ for every $i$.  Consequently each output
$\bm{y}_i$ lies in the convex hull of the value vectors
$\{\bm{v}_1,\dots,\bm{v}_n\}$.
```

```{admonition} Proof
:class: note
Positivity is immediate from the exponential, and the denominator of
Eq. (13.4) is exactly the sum of the numerators over $j$, so the
rows sum to one.  A combination with non-negative coefficients summing to one is
by definition a convex combination, and Eq. (13.2) exhibits
$\bm{y}_i$ as one.
```

The consequence is a genuine limitation.  A single attention layer cannot
produce an output outside the convex hull of its own values: it can select,
blend and interpolate, but it cannot extrapolate.  This is one reason the
feed-forward sublayer of Section *Multi-head attention and the transformer block* is not an optional extra --
without it a stack of attention layers would be confined to convex hulls
forever.

```{admonition} Theorem (Permutation equivariance)
:class: important
Let $\bm{\Pi}$ be an $n\times n$ permutation matrix.  Then self-attention
satisfies

$$
\mathrm{Attention}(\bm{\Pi}\bm{X})
  = \bm{\Pi}\,\mathrm{Attention}(\bm{X}).\tag{13.6}
$$

The output at a position depends on the *set* of tokens and on which token
sits at that position, but not on the order of the others.
```

```{admonition} Proof
:class: note
The three maps of Eq. (13.3) act on each row independently, so
$\bm{\Pi}\bm{X}$ produces $\bm{\Pi}\bm{Q}$, $\bm{\Pi}\bm{K}$ and
$\bm{\Pi}\bm{V}$.  The score matrix becomes
$(\bm{\Pi}\bm{Q})(\bm{\Pi}\bm{K})^{\mathsf{T}}/\sqrt{d_k}
=\bm{\Pi}\bm{S}\bm{\Pi}^{\mathsf{T}}$.  A row-wise softmax commutes with a
permutation of the rows, and permuting the columns of a row permutes the entries
of that row's softmax identically, so
$\mathrm{softmax}(\bm{\Pi}\bm{S}\bm{\Pi}^{\mathsf{T}})
=\bm{\Pi}\,\mathrm{softmax}(\bm{S})\,\bm{\Pi}^{\mathsf{T}}$.  Finally
$\bm{\Pi}\bm{A}\bm{\Pi}^{\mathsf{T}}\bm{\Pi}\bm{V}=\bm{\Pi}\bm{A}\bm{V}$, using
$\bm{\Pi}^{\mathsf{T}}\bm{\Pi}=\bm{I}$.
```

Compare Theorem thm:10-equi, which said convolution is the unique linear
map equivariant to *translation*.  Attention is equivariant to the much
larger group of *all* permutations, and a larger symmetry group means a
weaker model: attention knows nothing about order at all.  For a sentence or a
time series that is unacceptable, and Section *Masking and position* puts the
order back in by hand.

```{admonition} Proposition (Cost)
:class: important
Computing Eq. (13.5) requires $\bigO(n^{2}d_k+n^{2}d_v)$
arithmetic operations and $\bigO(n^{2})$ memory for the matrix $\bm{A}$.
```

```{admonition} Proof
:class: note
$\bm{Q}\bm{K}^{\mathsf{T}}$ is a product of an $n\times d_k$ and a $d_k\times n$
matrix, costing $\bigO(n^{2}d_k)$; the softmax is $\bigO(n^{2})$; and
$\bm{A}\bm{V}$ costs $\bigO(n^{2}d_v)$.
```

The quadratic scaling in sequence length is the central practical difficulty of
the architecture, and the reason for a large literature on sparse, low-rank and
kernelised approximations.  Contrast a convolution, which by
Eq. (10.16) touches only $F^{2}$ neighbours per position, and a
recurrent layer, which is linear in $n$ but strictly sequential and therefore
cannot be parallelised across time.  Attention buys global access and
parallelism, and pays for both in $n^{2}$.

### Why the \texorpdfstring{$1/\sqrt{d_k
$}{1/sqrt(dk)}}

The scaling in Eq. (13.5) looks like a detail.  It is not, and
the reason can be made precise.

```{admonition} Proposition (Variance of the logits)
:class: important
Let $\bm{q},\bm{k}\in\mathbb{R}^{d_k}$ have independent entries with mean zero
and variance $\sigma^{2}$.  Then

$$
\mathbb{E}\left[\bm{q}\cdot\bm{k}\right]=0,
  \qquad
  \var\left[\bm{q}\cdot\bm{k}\right]=d_k\sigma^{4},\tag{13.7}
$$

so the logits have standard deviation $\sigma^{2}\sqrt{d_k}$, growing without
bound with the head dimension.  Dividing by $\sqrt{d_k}$ makes the variance
independent of $d_k$.
```

```{admonition} Proof
:class: note
$\bm{q}\cdot\bm{k}=\sum_{r}q_rk_r$ is a sum of $d_k$ independent terms, each of
mean $\mathbb{E}[q_r]\mathbb{E}[k_r]=0$ and variance
$\mathbb{E}[q_r^{2}k_r^{2}]-0=\sigma^{4}$ by independence.  Variances of
independent variables add.
```

Measured with $\sigma^{2}=1$ over $4000$ samples:


```
  d_k    Var(q.k) unscaled   Var(q.k) scaled   prediction d_k
      4              4.22            1.0538             4
     16             15.87            0.9916            16
     64             61.48            0.9607            64
    256            251.83            0.9837           256
   1024           1011.47            0.9878          1024
```


The middle column tracks the last one over three decades, and the scaled variant
sits at one throughout.  Figure fig:attnscaling(a) plots it.

That is the statistics.  The reason it matters is what large logits do to the
softmax.  As the spread of the scores grows, the softmax approaches a hard
maximum: one weight goes to one, the rest to zero, and the derivative goes to
zero with them.  Measuring the Frobenius norm of the softmax Jacobian for
logits drawn from $\mathcal{N}(0,c^{2})$:


```
  scale   max A     entropy    ||d softmax/d s||_F
   0.25  0.06851    3.4214   4.808e-02
   1.00  0.37786    2.4937   1.029e-01
   4.00  0.98926    0.0703   8.223e-03
  16.00  1.00000    0.0000   1.745e-09
  64.00  1.00000    0.0000   3.276e-36
```


Between $c=1$ and $c=64$ the gradient falls by *thirty-five orders of
magnitude*.  A network whose attention weights sit in that regime receives no
gradient through them at all and cannot learn what to attend to.  Since
Proposition prop:13-variance says $c$ grows as $\sqrt{d_k}$, a model with
$d_k=64$ and unscaled logits starts life at $c\approx8$, already well into the
saturated region.  The $1/\sqrt{d_k}$ is what keeps it at $c\approx1$.

![a Variance of the attention logits against head dimension.  The unscal](../BookML/BookFigures/chapter13_transformers/attention_scaling.png)

*(a) Variance of the attention logits against head dimension.  The unscaled dot product follows the prediction $d_k$ of Proposition prop:13-variance exactly; dividing by $\sqrt{d_k}$ flattens it to one.  (b) Frobenius norm of the softmax Jacobian against the scale of the logits: past a scale of about four the softmax saturates and the gradient collapses, reaching $10^{-36}$ by scale $64$.  (c) An attention matrix $A_{ij}$ for an untrained network: every row sums to one, by Proposition prop:13-convex.*

```{admonition} The same failure as Chapter 11, at a different place
:class: tip
Section *Why long sequences are hard* showed gradients vanishing through a product of
Jacobians along a sequence.  Here they vanish through a single saturated
softmax.  Both are cured by keeping a quantity at scale one -- the spectral
radius there, the logit variance here -- and in both cases the cure is a
choice of scaling made before training rather than an adjustment made during it.
This is the recurring lesson of Chapters 8 to 13:
deep networks are trained by controlling scales.
```


## Masking and position

**Masks.** 
Sometimes a position must not see another.  In autoregressive generation, the
prediction at step $i$ may depend only on steps $j\le i$, or the model would be
allowed to read its own answer.  This is imposed by adding a *mask* to the
scores before the softmax,

$$
s_{ij} \;\longleftarrow\; s_{ij} + M_{ij},
  \qquad
  M_{ij} = \begin{cases}0, & j\le i,\\ -\infty, & j>i,\end{cases}\tag{13.8}
$$

so that $\exp(s_{ij})=0$ above the diagonal and $\bm{A}$ becomes lower
triangular.  The implementation adds $-\infty$ rather than deleting entries so
that the shape of the computation is unchanged.

A mask destroys Theorem thm:13-perm, and deliberately: the whole purpose
is to make position $i$ different from position $j$.  We verify both facts in
Section *Implementation and verification*.

**Positional encodings.** 

Theorem thm:13-perm says a transformer without further information cannot
distinguish a sentence from an anagram of it.  The standard remedy is to add a
position-dependent vector to each token before the first block.  The sinusoidal
choice of the original architecture is

$$
\mathrm{PE}(m,2r) = \sin\!\left(\frac{m}{10000^{2r/d}}\right),
  \qquad
  \mathrm{PE}(m,2r+1) = \cos\!\left(\frac{m}{10000^{2r/d}}\right),\tag{13.9}
$$

for position $m$ and channel $r$, and $\bm{x}_m \leftarrow \bm{x}_m
+\mathrm{PE}(m,\cdot)$.  The construction is a bank of sinusoids at
geometrically spaced frequencies, so that low channels vary slowly across the
sequence and high channels vary fast -- a positional analogue of the multiscale
argument of Section *Pooling*.

Its useful property is that the inner product
$\mathrm{PE}(m)\cdot\mathrm{PE}(m')$ depends mainly on $m-m'$ and decays with
it, so the encoding acts as a similarity kernel on positions and a
*relative* notion of distance is available to the dot product of
Eq. (13.4) without ever being written down.
Figure fig:attnmasks shows the encoding and that kernel.  Learned
positional embeddings and explicit relative encodings are the two common
alternatives.

![a An attention matrix under the causal mask of Eq. 13.8 the strict upp](../BookML/BookFigures/chapter13_transformers/attention_masks.png)

*(a) An attention matrix under the causal mask of Eq. (13.8): the strict upper triangle is exactly zero, measured to $0$.  (b) The sinusoidal positional encoding of Eq. (13.9), channels against position; low channels vary slowly, high channels fast. (c) The inner product of the encoding at position $32$ with every other position: a similarity kernel peaked on the diagonal, which is how relative distance reaches the dot product of Eq. (13.4).*


## Multi-head attention and the transformer block

One attention matrix imposes a single pattern of couplings.  Real problems have
several -- syntactic and semantic in language, local and long-range in a field.
*Multi-head* attention runs $H$ attentions in parallel with separate
projections and mixes the results:

$$
\mathrm{head}_h = \mathrm{Attention}\!\left(
    \bm{X}\bm{W}_Q^{(h)},\;\bm{X}\bm{W}_K^{(h)},\;\bm{X}\bm{W}_V^{(h)}\right),\tag{13.10}
$$

$$
\mathrm{MultiHead}(\bm{X})
  = \mathrm{Concat}\!\left(\mathrm{head}_1,\dots,\mathrm{head}_H\right)\bm{W}_O.\tag{13.11}
$$

Taking $d_k=d_v=d/H$ keeps the cost of $H$ heads equal to that of one head of
full width, so multiple heads are free.  Different heads reliably learn
different patterns, and inspecting them is the beginning of interpretability
work.

**The block.** 
Attention is one sublayer of a *transformer block*, which also contains a
position-wise feed-forward network, residual connections and normalisation.  In
the pre-norm arrangement now standard,

$$
\begin{split}
    \bm{X} &\;\longleftarrow\; \bm{X}
      + \mathrm{MultiHead}\!\left(\mathrm{LN}(\bm{X})\right),\\
    \bm{X} &\;\longleftarrow\; \bm{X}
      + \mathrm{MLP}\!\left(\mathrm{LN}(\bm{X})\right),
  \end{split}\tag{13.12}
$$

where the MLP is applied to each token separately,
$\mathrm{MLP}(\bm{z})=\bm{W}_2\,\mathrm{GELU}(\bm{W}_1\bm{z}+\bm{b}_1)+\bm{b}_2$,
with the GELU activation of Section *Activation functions*, and layer
normalisation acts across the feature axis of each token,

$$
\mathrm{LN}(\bm{z}) = \bm{\gamma}\odot
    \frac{\bm{z}-\mu(\bm{z})}{\sqrt{\sigma^{2}(\bm{z})+\epsilon}} + \bm{\beta},
  \qquad
  \mu(\bm{z})=\frac{1}{d}\sum_{r}z_r .\tag{13.13}
$$

Each ingredient earns its place.  The MLP supplies the non-linearity that
Proposition prop:13-convex showed attention alone lacks, and it is where
most of the parameters live.  The residual connections give the gradient an
undamped path from the loss to every layer -- precisely the additive path that
Eq. (11.26) gave the LSTM cell state.  Layer normalisation keeps
the scale of the activations fixed, which by Section *Why the \texorpdfstring{$1/\sqrt{d_k}$}{1/sqrt(dk)}* is what
the softmax needs.

A transformer is a stack of these blocks.  Nothing in it is unfamiliar: it is
attention, plus the multilayer perceptron of Chapter 8, plus
residual connections and normalisation.


## Implementation and verification

We write attention with autograd rather than by hand, for the reason
Section *Automatic differentiation* gave: differentiating a softmax nested inside two
matrix products is exactly the error-prone drudgery automatic differentiation
exists to remove.


In [ ]:
def softmax_rows(S):
    """Row-wise softmax, shifted for stability."""
    S = S - np.max(S, axis=-1, keepdims=True)
    E = np.exp(S)
    return E / np.sum(E, axis=-1, keepdims=True)


def attention(Q, K, V, mask=None):
    """Scaled dot-product attention, Eq. (13.attention).

    Q is (n, d_k), K is (m, d_k), V is (m, d_v); the output is (n, d_v).
    """
    d_k = Q.shape[-1]
    S = Q @ K.T / np.sqrt(d_k)                 # (n, m) scores
    if mask is not None:
        S = S + mask                           # -inf where attention is banned
    A = softmax_rows(S)                        # (n, m), rows sum to one
    return A @ V, A


Multi-head attention loops over heads, concatenates and mixes; the block adds
the residuals and normalisation of Eq. (13.12).


In [ ]:
def block(P, X, mask=None):
    """Pre-norm transformer block, Eq. (13.block).

    X -> X + MHA(LN(X)) -> X + MLP(LN(X)).  The residual paths carry the
    identity, which is what keeps deep stacks trainable (cf. Section 11.lstmwhy).
    """
    Y, A = multihead(P, layernorm(X, P["g1"], P["be1"]), mask)
    X = X + Y
    Z = layernorm(X, P["g2"], P["be2"])
    X = X + gelu(Z @ P["W1"] + P["b1"]) @ P["W2"] + P["b2"]
    return X, A


The checks follow the propositions one by one.


```
rows of A sum to 1 : 2.22e-16
A >= 0             : True
head output within the bounding box of V : True

=== permutation equivariance:  Att(PX) = P Att(X) ===
  trial 0: max|Att(PX) - P Att(X)| = 1.67e-16
  trial 1: max|Att(PX) - P Att(X)| = 3.33e-16
  trial 2: max|Att(PX) - P Att(X)| = 2.22e-16

=== a causal mask breaks it (as it must) ===
  max|Att(PX) - P Att(X)| with mask = 1.897e+00
  attention matrix upper triangle (should be 0): 0.00e+00
```


Proposition prop:13-convex and Theorem thm:13-perm hold to machine
precision, and the mask does exactly what Eq. (13.8) says: it
zeroes the upper triangle exactly and destroys the equivariance completely.
The batched implementation used for training agrees with the looped one to
$8.9\times10^{-16}$.


## What one attention layer cannot do

Theorem thm:13-perm and Proposition prop:13-convex are statements
about what attention *is*.  This section is about what a given depth of it
can *compute*, and the answer is sharper than one might guess.

Consider *associative recall*.  A sequence presents $L$ key-value pairs
$k_1v_1k_2v_2\cdots k_Lv_L$ and then repeats one of the keys; the model must
emit the value that followed it.  The task is trivial for a human, has an exact
algorithm, and requires reaching a specific earlier position whose distance
grows with $L$.  It is the cleanest test of global access there is.

Now look at what one attention layer can do with it.  The query at the final
position carries the repeated key, and the dot product of
Eq. (13.4) can certainly make it match the earlier position
holding the same key.  But matching that position retrieves *that
position's* value vector -- the key -- and what is wanted is the token
*one place to the right*.  A single layer has no way to attend to one
position and read from another.

It needs two.  The first layer copies each token's predecessor into its
representation; the second matches the query against keys and now finds the
value already attached.  This composition is known as an *induction head*,
and it is a mechanism rather than a metaphor: the prediction is that one block
fails and two succeed.  We test it, with a vocabulary of eight symbols, so that
chance is $0.125$.

| \noalign{}
$L$ | Model | Parameters | Test accuracy |
|---|---|---|---|
| \noalign{}\noalign{}
$2$ | transformer, 1 block | $8928$ | $1.000$ |
| $2$ | RNN (Chapter 11) | $8858$ | $1.000$ |
| \noalign{}\noalign{}
$4$ | transformer, 1 block | $8928$ | $0.366$  ($0.352$--$0.380$) |
| $4$ | RNN (Chapter 11) | $8858$ | $0.351$  ($0.333$--$0.370$) |
| $4$ | transformer, 2 blocks | $17344$ | $\mathbf{0.993}$  ($0.985$--$1.000$) |
| \noalign{} |

*Associative recall with $L$ key-value pairs, so sequence length
$T=2L+1$.  All models trained by Adam with $\eta=3\times10^{-3}$ on freshly
generated batches; accuracy on $400$ unseen sequences, ranges over two seeds.
Chance is $0.125$.*

At $L=2$ the sequence is five tokens long and everything works.  At $L=4$ both
the one-block transformer and the recurrent network of Chapter 11
sit at about $0.36$, well above chance but nowhere near the task, and they sit
there for three thousand updates.  Adding a second block takes the same
architecture to $0.993$.

This is worth dwelling on, because it cuts against the usual way of describing
transformers.  Global access is *not* sufficient.  The one-block model can
already see every position -- Proposition prop:13-cost charges it $n^{2}$
for the privilege -- and it still cannot solve a task that a two-line program
solves, because the computation it needs is a composition of two retrievals and
it has only one layer in which to do them.  Depth is not a way of adding
capacity here; it is a way of adding *steps*.

```{admonition} An honest reading of the middle rows
:class: tip
The one-block transformer does
not beat the recurrent network on this task; the two are statistically
indistinguishable at $0.366$ against $0.351$.  A comparison stopped there would
conclude that attention offers nothing.  The right conclusion is that the
architecture was mismatched to the computation, and the fix was depth rather
than width or attention span.  When a model plateaus well above chance, it is
worth asking what algorithm it would need and how many sequential steps that
algorithm takes.
```


## Comparisons and interpretations

### Against the other architectures

The four architectures of Chapters 8 to 13 differ in
exactly one respect: which elements interact, and how strongly.

| \noalign{}
Architecture | Coupling $y_i=\sum_j C_{ij}x_j$ | Cost per layer |
|---|---|---|
| \noalign{}\noalign{}
MLP | $C_{ij}=W_{ij}$, dense and fixed | $\bigO(n^{2}d^{2})$ |
| CNN | $C_{ij}=w_{i-j}$, local and fixed | $\bigO(nF d^{2})$ |
| RNN | $C$ lower triangular, applied step by step | $\bigO(nd^{2})$, sequential |
| Transformer | $C_{ij}=A_{ij}(\bm{X})$, dense and *adaptive* | $\bigO(n^{2}d)$ |
| \noalign{} |

An MLP has fixed weights $\bm{W}$ that do not depend on the input.  A
convolution is the special case in which those weights are local and shared, by
Theorem thm:10-toeplitz.  A recurrent layer reaches distant positions only
by composing many short steps, which is what
Theorem thm:11-vanishing punishes.  Attention alone lets the coupling
matrix be a *function of the data*:

$$
\text{fixed couplings}\quad\longrightarrow\quad
  \text{couplings }A_{ij}(\bm{X})\text{ computed at run time}.\tag{13.14}
$$

That is the whole of the innovation, and everything else in the architecture is
scaffolding to make it trainable.

### A statistical-mechanics reading

Equation (13.4) will look familiar to anyone who has met a
canonical ensemble.  Writing $E_{ij}=-s_{ij}$,

$$
A_{ij} = \frac{e^{-E_{ij}}}{\sum_{\ell}e^{-E_{i\ell}}}
         = \frac{e^{-E_{ij}}}{Z_i},\tag{13.15}
$$

which is a Gibbs distribution over interaction partners at inverse temperature
one, with $Z_i$ a per-query partition function.  The scores are effective
energies; the softmax is a local Boltzmann weighting; and the $1/\sqrt{d_k}$ of
Section *Why the \texorpdfstring{$1/\sqrt{d_k}$}{1/sqrt(dk)}* is a choice of temperature, with saturation being the
zero-temperature limit in which the distribution collapses onto its ground
state.

The aggregation step then reads

$$
\bm{y}_i = \sum_j J_{ij}(\bm{X})\,\bm{v}_j,
  \qquad J_{ij}\equiv A_{ij},\tag{13.16}
$$

which is a mean-field update with *state-dependent* couplings.  In an
Ising or Hopfield model the $J_{ij}$ are fixed by the Hamiltonian; here they are
recomputed from the current configuration at every layer.  A transformer is,
in this reading, a nonlinear adaptive mean-field model, and the analogy is close
enough to have been productive: the connection between attention and modern
Hopfield networks with exponential capacity is more than superficial.

### Kernels and graphs

Equation (13.2) also reads as a discretised integral operator,

$$
\bm{y}_i = \sum_j K(\bm{x}_i,\bm{x}_j)\,\bm{v}_j
  \qquad\longleftrightarrow\qquad
  (\mathcal{K}f)(x) = \int K(x,x')f(x')\,\mathrm{d}x',\tag{13.17}
$$

with the difference that $K$ is learned rather than prescribed.  Compare the
kernel methods of Chapter 6, where $K$ was chosen in advance and
the representer theorem then fixed everything: attention learns the kernel and
gives up the theory.

Viewed as a graph, attention defines a complete weighted graph on the tokens
with edge weights $A_{ij}$, which makes a transformer a graph neural network
whose graph is inferred rather than given.  A message-passing network on a fixed
graph is the special case in which $A_{ij}$ is constrained to the adjacency
structure.

### Why this matters for differential equations

Chapters 9 and 10 both met the same obstacle from
different sides.  A convolutional network handles local structure superbly and
reaches distant parts of a domain only by stacking layers, at the rate given by
Proposition prop:10-rf.  But elliptic equations are *globally*
coupled -- change a boundary value and the solution changes everywhere at once
-- and so are nonlocal operators, multiscale dynamics and problems with global
constraints.  Attention couples the whole domain in one layer.

The natural framing is operator learning.  Rather than solving one problem, as
in Chapter 9, one learns the solution operator

$$
\mathcal{G}: a(x)\;\longmapsto\;u(x),
  \qquad\text{or}\qquad
  \mathcal{G}: u_0(x)\;\longmapsto\;u(x,t),\tag{13.18}
$$

mapping a coefficient field or an initial condition to a solution.  Discretise
the field at points $x_1,\dots,x_n$ and let each token carry what is known at
that point,

$$
\bm{x}_i^{\mathrm{token}}
   = \bigl(x_i,\;u(x_i),\;a(x_i),\dots\bigr),\tag{13.19}
$$

and Eq. (13.17) becomes a learned Green's function: $A_{ij}$ is how
much the solution at $x_i$ depends on the data at $x_j$.  For the Poisson
equation of Section *The one-dimensional Poisson equation* that dependence is the actual Green's
function, and a trained attention matrix can be compared against it directly --
an unusually direct check on what such a model has learned, and one worth doing.

The competitor is the Fourier neural operator, which achieves nonlocality by
multiplication in the frequency domain, in the spirit of the notebox of
Section *Toeplitz structure*.  Fourier methods are natural for translation-invariant
problems on regular grids and cost $\bigO(n\log n)$; attention handles irregular
domains and learns data-dependent kernels, and costs $\bigO(n^{2})$.  Which wins
depends on whether the problem's structure is known in advance -- the same
question that decided between hard and soft constraints in
Section *Physics-informed neural networks*.


## PyTorch and TensorFlow

PyTorch supplies attention at three levels.  The lowest is
\verb!scaled_dot_product_attention!, which is Eq. (13.5) and
nothing else; above it \verb!nn.MultiheadAttention! adds the projections of
Eq. (13.11); above that \verb!nn.TransformerEncoderLayer! is the
whole of Eq. (13.12).  Writing the block explicitly is more
instructive:


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class Block(nn.Module):
    """Eq. (13.block), pre-norm, written out rather than assembled."""
    def __init__(self, d=64, H=4, d_ff=256, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)                  # Eq. (13.layernorm)
        self.attn = nn.MultiheadAttention(d, H, dropout=dropout,
                                          batch_first=True)
        self.ln2 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(
            nn.Linear(d, d_ff), nn.GELU(), nn.Linear(d_ff, d),
            nn.Dropout(dropout))

    def forward(self, x, mask=None):
        h = self.ln1(x)
        a, A = self.attn(h, h, h, attn_mask=mask, need_weights=True)
        x = x + a                                   # residual: see Section 13.block
        x = x + self.mlp(self.ln2(x))
        return x, A


def causal_mask(n, device=None):
    """Eq. (13.mask): True where attention is forbidden."""
    return torch.triu(torch.ones(n, n, dtype=torch.bool, device=device), 1)


class Transformer(nn.Module):
    def __init__(self, n_vocab, d=64, H=4, d_ff=256, n_blocks=2, n_ctx=64):
        super().__init__()
        self.emb = nn.Embedding(n_vocab, d)
        self.pos = nn.Parameter(torch.zeros(n_ctx, d))   # learned; or Eq. (13.posenc)
        self.blocks = nn.ModuleList(
            [Block(d, H, d_ff) for _ in range(n_blocks)])
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, n_vocab)

    def forward(self, idx, causal=True):
        n = idx.shape[1]
        x = self.emb(idx) + self.pos[:n]
        m = causal_mask(n, idx.device) if causal else None
        for blk in self.blocks:
            x, _ = blk(x, m)
        return self.head(self.ln_f(x))


Three details are worth flagging.  \verb!batch_first=True! is not the default
and the alternative layout catches everyone once.  \verb!attn_mask! expects
\verb!True! where attention is *forbidden*, the opposite of what the name
suggests to many readers.  And the parameter count of a block is
$4d^{2}$ for the attention plus $2dd_{\mathrm{ff}}$ for the MLP, so with the
usual $d_{\mathrm{ff}}=4d$ the feed-forward part holds two thirds of the
parameters -- attention is where the computation is, not where the parameters
are.

The same in Keras, where \verb!MultiHeadAttention! plays the same role:


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers


class Block(layers.Layer):
    """Eq. (13.block) in Keras; note key_dim is d_k, not d."""
    def __init__(self, d=64, H=4, d_ff=256, dropout=0.1):
        super().__init__()
        self.ln1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn = layers.MultiHeadAttention(num_heads=H, key_dim=d // H,
                                              dropout=dropout)
        self.ln2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = tf.keras.Sequential([
            layers.Dense(d_ff, activation="gelu"),
            layers.Dense(d), layers.Dropout(dropout)])

    def call(self, x, training=False):
        h = self.ln1(x)
        x = x + self.attn(h, h, h, use_causal_mask=True, training=training)
        return x + self.mlp(self.ln2(x), training=training)


def build(n_vocab, n_ctx=64, d=64, H=4, d_ff=256, n_blocks=2):
    inp = layers.Input(shape=(n_ctx,), dtype="int32")
    tok = layers.Embedding(n_vocab, d)(inp)
    pos = layers.Embedding(n_ctx, d)(tf.range(n_ctx))
    x = tok + pos
    for _ in range(n_blocks):
        x = Block(d, H, d_ff)(x)
    out = layers.Dense(n_vocab)(layers.LayerNormalization(epsilon=1e-6)(x))
    model = tf.keras.Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(3e-4),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))
    return model


## Strengths, weaknesses and what to remember

A transformer replaces the fixed interaction patterns of the earlier
architectures by couplings computed from the data, Eq. (13.14).
Everything else -- the multilayer perceptron, the residual connections, the
normalisation -- is machinery from earlier chapters, assembled to make that one
idea trainable.

The properties are sharp and we proved them.  Attention averages, so its output
lies in the convex hull of its values (Proposition prop:13-convex) and it
cannot extrapolate without the feed-forward sublayer.  It is equivariant to
*all* permutations (Theorem thm:13-perm), a much larger symmetry group
than the translations of Theorem thm:10-equi, which is why position has to
be added by hand.  It costs $\bigO(n^{2}d)$ (Proposition prop:13-cost),
which is the price of global access and the reason for a large literature on
approximating it.

The $1/\sqrt{d_k}$ is not cosmetic.  Proposition prop:13-variance predicts
$\var(\bm{q}\cdot\bm{k})=d_k$, measured as $1011$ at $d_k=1024$; and the softmax
Jacobian falls by thirty-five orders of magnitude between logit scale $1$ and
scale $64$.  Without the scaling a wide head starts training with no usable
gradient through its attention weights at all.

The result to carry away is Section *What one attention layer cannot do*.  On associative recall
at $L=4$, a one-block transformer reached $0.366$ and a matched recurrent
network $0.351$ -- indistinguishable, and both far from the task.  Two blocks
reached $0.993$.  Seeing every position is not the same as being able to compose
two retrievals, and depth here supplies *steps* of computation rather than
capacity.  Global access is necessary and not sufficient.

Set against the strengths -- global context, full parallelism across positions,
excellent scaling with data and model size, a learned rather than assumed
interaction structure -- are real weaknesses: quadratic cost in sequence length,
large data requirements, no inductive bias whatever about locality or order, and
an interpretability that is better than a dense layer's but far from complete.
The architecture is a good default when the structure of the interactions is
unknown and the data are plentiful.  When the structure *is* known --
locality in an image, causality in a time series, translation invariance in a
homogeneous medium -- a convolution or a recurrence encodes it for free, and
free structure beats learned structure whenever the structure is right.

The programs are in the directory  

`doc/BookML/BookPrograms/chapter13_transformers`.  

Every listing above appears there as a numbered file, and three modules run
start to finish and reproduce the numbers quoted in the text:

- `attention.py` -- scaled dot-product attention, multi-head
   attention, layer normalisation, the block of
   Eq. (13.12), causal masks, sinusoidal positional
   encodings, and batched versions of all of them.
- `verify_attention.py` -- row-stochasticity, the convex-hull
   property, permutation equivariance and its destruction by a mask, the
   logit-variance table, the softmax-saturation table, and the agreement
   between the looped and batched implementations.
- `recall.py` and `run_recall.py` -- the associative-recall
   task of Table 13.1, with the one-block and two-block
   transformers and the Chapter 11 recurrent baseline.

The figures are generated by `ch13_figures.py` in
`doc/BookML/BookFigures`; neither is drawn by hand.


## Exercises

### Warm-up exercises

1. **Shapes and costs.**
   With $n=512$, $d=768$, $H=12$ and $d_k=d/H$:
   (a) what are the shapes of $\bm{Q}$, $\bm{K}$, $\bm{V}$, $\bm{A}$ and the
   output of one head?
   (b) How many parameters has one block with $d_{\mathrm{ff}}=4d$, split
   between attention and the MLP?
   (c) How much memory does $\bm{A}$ need in single precision, for all heads at
   once, and what happens at $n=8192$?
2. **Permutation equivariance.**
   Prove Theorem thm:13-perm for multi-head attention, and then show that
   adding the positional encoding of Eq. (13.9) destroys it.  Is
   the composition equivariant to any group at all?
3. **The convex hull.**
   Verify Proposition prop:13-convex numerically, then construct a target
   output that a single attention layer provably cannot produce, and confirm that
   adding the MLP of Eq. (13.12) makes it reachable.
4. **Temperature.**
   Replace $1/\sqrt{d_k}$ by $\beta$ in Eq. (13.4) and study the
   limits $\beta\to0$ and $\beta\to\infty$.  Show that the first gives uniform
   averaging and the second a hard $\arg\max$, and relate both to
   Eq. (13.15).  What is the entropy of the attention distribution in
   each limit?
5. **Positional encodings.**
   Show that $\mathrm{PE}(m+\delta)$ is a linear function of $\mathrm{PE}(m)$ for
   fixed $\delta$, so that relative position is linearly accessible.  Then plot
   $\mathrm{PE}(m)\cdot\mathrm{PE}(m')$ and confirm that it depends mainly on
   $m-m'$.
6. **Masks.**
   Write the mask for (a) causal attention, (b) attention restricted to a window
   of width $w$, (c) attention that ignores padding tokens.  For (b), show that
   the layer becomes a convolution with a data-dependent kernel, and relate this
   to Theorem thm:10-toeplitz.

### Project-style exercise: a transformer from scratch

**Part a: the machinery.** 
Implement scaled dot-product attention, multi-head attention, layer
normalisation and the block of Eq. (13.12).  Verify
row-stochasticity, the convex-hull property and permutation equivariance to
machine precision, and check every gradient against finite differences.
Implement the batched version and verify it against the looped one.

**Part b: the scaling.** 
Reproduce Proposition prop:13-variance and the saturation table of
Section *Why the \texorpdfstring{$1/\sqrt{d_k}$}{1/sqrt(dk)}*.  Then train the same model with and without the
$1/\sqrt{d_k}$ at several values of $d_k$, and report at what $d_k$ the unscaled
version stops training at all.  Plot the entropy of the attention distribution
during training in both cases.

**Part c: depth versus width.** 
Reproduce Table 13.1.  Then separate the two explanations for the
one-block failure: is it depth, or capacity?  Train a one-block model with the
width increased until its parameter count matches the two-block model, and
report what happens.  Then extend the table to $L=8$ and $L=16$ and plot
accuracy against $L$ for one block, two blocks and the recurrent baseline.

**Part d: reading the attention.** 
For a trained two-block model, plot the attention matrices of both blocks on
several test sequences.  Can you identify the mechanism described in
Section *What one attention layer cannot do* -- one head copying the predecessor, another
matching the query?  Quantify it: measure how much of the first block's
attention mass sits on the diagonal offset by one.

**Part e: against the alternatives.** 
Compare the transformer, the convolutional network of Chapter 10 and
the recurrent network of Chapter 11 on the same task at matched
parameter counts, measuring accuracy, wall-clock time per update and memory.
Then repeat on a task with genuinely local structure and report which
architecture wins there.  The two answers should differ, and the discussion of
why is the point of the exercise.

**Part f: a learned Green's function.** 
Solve the one-dimensional Poisson equation of Section *The one-dimensional Poisson equation* for many
right-hand sides, and train an attention model to map the discretised source
$f(x_j)$ to the solution $u(x_i)$, as in Eq. (13.19).  Compare
the learned attention matrix against the exact Green's function of the operator.
How close is it, and does it get closer with more data or with more blocks?

**Part g: the cost.** 
Measure the wall-clock time and peak memory of your attention implementation as
$n$ grows, and confirm Proposition prop:13-cost.  Then implement one
approximation -- a sliding window, a random sparse pattern, or a low-rank
factorisation of $\bm{A}$ -- and report what it costs in accuracy on the recall
task of part c.
